In [1]:
import polars as pl

## Loading and Inspecting Data

### Main Dataframe (here Finland only)
Demonstrates use of:
- col
- read_parquet
- select
- filter
- schema
- head
- describe
- height
- width
- shape

In [2]:
columns_of_interest = [
    pl.col("entity_content"),
    pl.col("entity_category"),
    pl.col("sentence_content_current"),
    pl.col("sentence_content_previous"),
    pl.col("sentence_content_next"),
    pl.col("country"),
    pl.col("entity_word_id"),
    pl.col("session_date"),
    pl.col("speaker_id"),
    pl.col("speaker_ana"),
    pl.col("sentence_sentiment_value"),
    pl.col("debate_topic")
]

In [3]:
df = pl.read_parquet("../data/facts/FIN*.parquet").select(
    columns_of_interest
).filter(
    pl.col("speaker_ana") == "regular"
)
df.schema

Schema([('entity_content', String),
        ('entity_category', String),
        ('sentence_content_current', String),
        ('sentence_content_previous', String),
        ('sentence_content_next', String),
        ('country', String),
        ('entity_word_id', String),
        ('session_date', Date),
        ('speaker_id', String),
        ('speaker_ana', String),
        ('sentence_sentiment_value', Float32),
        ('debate_topic', String)])

In [4]:
df.head()

entity_content,entity_category,sentence_content_current,sentence_content_previous,sentence_content_next,country,entity_word_id,session_date,speaker_id,speaker_ana,sentence_sentiment_value,debate_topic
str,str,str,str,str,str,str,date,str,str,f32,str
"""Small Parliament""","""LOC""","""Members of the large committee…","""Mr President !""",null,"""FIN""","""ParlaMint-FI_2015-05-05-ps-2.s…",2015-05-05,"""SirkkaLiisaAnttila""","""regular""",2.456,"""other"""
"""Constitutional Committee""","""ORG""","""The members of the Constitutio…","""Mr. Speaker !""",null,"""FIN""","""ParlaMint-FI_2015-05-05-ps-2.s…",2015-05-05,"""PerttiSalolainen""","""regular""",2.39,"""other"""
"""Committee on Foreign Affairs""","""ORG""","""The members of the Committee o…","""Mr. Speaker !""",null,"""FIN""","""ParlaMint-FI_2015-05-05-ps-2.s…",2015-05-05,"""SeppoKääriäinen""","""regular""",2.426,"""mixed"""
"""Committee""","""ORG""","""The members of the Committee o…","""Mr. Speaker !""",null,"""FIN""","""ParlaMint-FI_2015-05-05-ps-2.s…",2015-05-05,"""SeppoKääriäinen""","""regular""",2.426,"""mixed"""
"""Small Parliament""","""ORG""","""The members of the Committee o…","""Mr. Speaker !""",null,"""FIN""","""ParlaMint-FI_2015-05-05-ps-2.s…",2015-05-05,"""SeppoKääriäinen""","""regular""",2.426,"""mixed"""


In [5]:
df.describe()

statistic,entity_content,entity_category,sentence_content_current,sentence_content_previous,sentence_content_next,country,entity_word_id,session_date,speaker_id,speaker_ana,sentence_sentiment_value,debate_topic
str,str,str,str,str,str,str,str,str,str,str,f64,str
"""count""","""39210""","""39210""","""39210""","""39004""","""35549""","""39210""","""39210""","""39210""","""39210""","""39210""",39210.0,"""39210"""
"""null_count""","""0""","""0""","""0""","""206""","""3661""","""0""","""0""","""0""","""0""","""0""",0.0,"""0"""
"""mean""",null,null,null,null,null,null,null,"""2015-10-11 12:13:26.488141""",null,null,2.181256,null
"""std""",null,null,null,null,null,null,null,null,null,null,1.62246,null
"""min""","""-American""","""LOC""",""""" Abbreviation of the time lim…",""""" Abbreviation of the time lim…",""""" Abbreviation of the time lim…","""FIN""","""ParlaMint-FI_2015-05-05-ps-2.s…","""2015-05-05""","""AilaPaloniemi""","""regular""",-0.146,"""argic"""
"""25%""",null,null,null,null,null,null,null,"""2015-09-23""",null,null,0.49,null
"""50%""",null,null,null,null,null,null,null,"""2015-10-22""",null,null,2.297,null
"""75%""",null,null,null,null,null,null,null,"""2015-12-08""",null,null,3.462,null
"""max""","""ärade""","""PER""","""— When Mr Terho spoke of the f…","""♪ We are ♪""","""♪ We are ♪""","""FIN""","""ParlaMint-FI_2015-12-18-ps-85.…","""2015-12-18""","""WilleRydman""","""regular""",5.739,"""welfa"""


In [6]:
print(f"{df.height} rows, {df.width} columns – shape: {df.shape}")

39210 rows, 12 columns – shape: (39210, 12)


### Joined Metadata Frame
Demonstrates use of:
- join
- filter
- exclude
- select

In [7]:
df_meta = pl.read_parquet("../data/Master_People.parquet"
                          ).join(
    pl.read_parquet("../data/Master_Affiliations.parquet").filter(
        (pl.col("start_date") < "2016") # note that the dates here are str (due to different formats present)
        & (pl.col("start_date") > "2014")
    ),
    on=["speaker_id","country"],
    how="inner"
).join(
    pl.read_parquet("../data/Master_Organizations.parquet").select(
        pl.exclude(["country"])
    ),
    on="org_id",
    how="inner"
).filter(
    [
        (pl.col("country") == "FIN")
    ]
)
df_meta.schema

Schema([('country', String),
        ('speaker_id', String),
        ('speaker_name', String),
        ('speaker_birth_year', String),
        ('speaker_gender', String),
        ('speaker_place_of_birth', String),
        ('org_id', String),
        ('role', String),
        ('start_date', String),
        ('end_date', String),
        ('org_name', String),
        ('org_role', String),
        ('party_ches_id', String),
        ('party_left_right_orientation', String)])

In [8]:
df_meta.head()

country,speaker_id,speaker_name,speaker_birth_year,speaker_gender,speaker_place_of_birth,org_id,role,start_date,end_date,org_name,org_role,party_ches_id,party_left_right_orientation
str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""FIN""","""MarkusMustajärvi""","""Mustajärvi Markus""","""1963""","""M""",null,"""fi_parliament""","""member""","""2015-04-23""",null,"""Suomen eduskunta""","""parliament""",null,null
"""FIN""","""MarkusMustajärvi""","""Mustajärvi Markus""","""1963""","""M""",null,"""party.VAS""","""member""","""2015-04-23""",null,"""Vasemmistoliitto""","""politicalParty""","""1404""","""orientation.L"""
"""FIN""","""SamiSavio""","""Savio Sami""","""1975""","""M""",null,"""fi_parliament""","""member""","""2015-04-22""",null,"""Suomen eduskunta""","""parliament""",null,null
"""FIN""","""SamiSavio""","""Savio Sami""","""1975""","""M""",null,"""party.PS""","""member""","""2015-04-22""",null,"""Pozitivna Slovenija""","""parliamentaryGroup""","""2914""","""orientation.CCL"""
"""FIN""","""SamiSavio""","""Savio Sami""","""1975""","""M""",null,"""party.PS""","""member""","""2015-04-22""",null,"""Perussuomalaiset""","""politicalParty""","""1405""","""orientation.RRF"""


In [9]:
df_meta.describe()

statistic,country,speaker_id,speaker_name,speaker_birth_year,speaker_gender,speaker_place_of_birth,org_id,role,start_date,end_date,org_name,org_role,party_ches_id,party_left_right_orientation
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""count""","""936""","""936""","""936""","""936""","""936""","""0""","""936""","""936""","""936""","""818""","""936""","""936""","""108""","""133"""
"""null_count""","""0""","""0""","""0""","""0""","""0""","""936""","""0""","""0""","""0""","""118""","""0""","""0""","""828""","""803"""
"""mean""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""std""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""FIN""","""AlexanderStubb""","""Aalto Touko""","""1942""","""F""",null,"""GOV""","""member""","""2014-01-01""","""2014-06-23""","""Bundesregierung der Republik Ö…","""government""","""1401""","""orientation.C"""
"""25%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""FIN""","""YmpäristöjailmastoministeriKri…","""Yanar Ozan""","""1991""","""M""",null,"""party.VIHR""","""minister""","""2015-11-06""","""2021-09-07""","""Κυβέρνηση της Ελληνικής Δημοκρ…","""politicalParty""","""3102""","""orientation.RRF"""


## Munging Data
Demonstrates:
- group_by
- agg
- len
- sort
- alias
- eval
- starts_with
- mean
- element
- value_counts
- get_column
- item
- strip_chars
- contains

Plus namespace functions:
- str
- list
- struct

In [10]:
df.group_by([pl.col("entity_category"), pl.col("entity_content")]).agg(
    pl.len()
).sort("len", descending=True)

entity_category,entity_content,len
str,str,u32
"""LOC""","""Finland""",5119
"""MISC""","""EUR""",2190
"""MISC""","""Finnish""",1869
"""MISC""","""Finns""",742
"""ORG""","""Government Programme""",654
…,…,…
"""LOC""","""Utsjoki Nuorgam""",1
"""ORG""","""Minister for the Environment a…",1
"""LOC""","""Raahe""",1


In [11]:
df.filter(
    pl.col("entity_content").str.starts_with("Swe")
).group_by(
    ["entity_content", "entity_category"],
    maintain_order=True
).agg(
    pl.len().alias("count"),
    pl.col("sentence_sentiment_value").mean().alias("mean_sentiment"),
    pl.col("debate_topic")
).with_columns(
    pl.col("debate_topic").list.sort().list.eval(pl.element().value_counts()
                                                 ).list.eval(
        pl.element().sort_by(pl.element().struct.field("count"), descending=True)
    ).alias("debate_topic_records")
)

entity_content,entity_category,count,mean_sentiment,debate_topic,debate_topic_records
str,str,u32,f32,list[str],list[struct[2]]
"""Sweden""","""LOC""",393,2.133595,"[""immig"", ""inter"", … ""housi""]","[{""macro"",66}, {""inter"",53}, … {""housi"",1}]"
"""Swedish""","""MISC""",123,2.222715,"[""defen"", ""macro"", … ""envir""]","[{""educa"",28}, {""immig"",16}, … {""housi"",1}]"
"""Swedes""","""MISC""",6,2.658667,"[""macro"", ""envir"", … ""domes""]","[{""envir"",2}, {""macro"",1}, … {""domes"",1}]"
"""Swedish People 's Party""","""ORG""",2,4.323,"[""mixed"", ""welfa""]","[{""welfa"",1}, {""mixed"",1}]"
"""Swedish Parliamentary Group""","""ORG""",3,3.463333,"[""mixed"", ""macro"", ""labor""]","[{""macro"",1}, {""labor"",1}, {""mixed"",1}]"
"""Swedish National Audit Office""","""ORG""",2,0.5365,"[""healt"", ""civil""]","[{""civil"",1}, {""healt"",1}]"


In [12]:
df.get_column("sentence_content_current")

sentence_content_current
str
"""Members of the large committee…"
"""The members of the Constitutio…"
"""The members of the Committee o…"
"""The members of the Committee o…"
"""The members of the Committee o…"
…
"""It is up to us to decide wheth…"
"""To be more precise , I intende…"
"""In paragraph 7 , I was meant t…"


In [13]:
df.filter(
    (pl.col("entity_category") == "LOC")
    &
    (pl.col("entity_content").str.contains(r"[Rr]ussia")) # Regex is supported
).select(
    # We're creating a new column from several existing columns
    (
            pl.col("sentence_content_previous").fill_null("")
            + " "
            + pl.col("sentence_content_current")
            + " "
            + pl.col("sentence_content_next").fill_null("")
    ).str.strip_chars().alias("sentence_with_context")
).item(10,"sentence_with_context")

"It is real and , as I said , in practice we do not know what the actual number of recruits is and what their activity is . Nor has the crisis between Russia and Ukraine subsided , but the unrest , as well as Russia 's certain isolation , are obvious all the time and , as the President - in - Office mentioned , Russia is not taking part in the Council of Europe 's activities as before , there is no similar negotiating link . We are also talking about a change in the security environment ."

In [14]:
df.get_column("entity_content")

entity_content
str
"""Small Parliament"""
"""Constitutional Committee"""
"""Committee on Foreign Affairs"""
"""Committee"""
"""Small Parliament"""
…
"""Finland"""
"""Section 35"""
"""Metsähallitus"""
